# Build target with min_30_inst

This notebook reads `user_overdue_5.13.xlsx` and `order_relation_5.13.xlsx`, rebuilds bill order by root order chain, and creates order-chain-level `target` using `min_30_inst`.

In [1]:
import pandas as pd
from pathlib import Path

# Works whether Jupyter starts from project root or from src.
CWD = Path.cwd()
DATA_DIR = CWD / 'data'
if not DATA_DIR.exists():
    DATA_DIR = CWD.parent / 'data'

OVERDUE_FILE = DATA_DIR / 'user_overdue.xlsx'
RELATION_FILE = DATA_DIR / 'order_relation.xlsx'
OUTPUT_FILE = DATA_DIR / 'user_target_min_30_inst.xlsx'

MOB_DAYS = 180
DPD_DAYS = 30
DATA_CUTOFF = pd.Timestamp('2026-05-11')

print('overdue file:', OVERDUE_FILE.name)
print('relation file:', RELATION_FILE.name)
print('output file:', OUTPUT_FILE.name)

overdue file: user_overdue.xlsx
relation file: order_relation.xlsx
output file: user_target_min_30_inst.xlsx


In [2]:
df_1 = pd.read_excel(OVERDUE_FILE)
df_relation = pd.read_excel(RELATION_FILE)

print('overdue shape:', df_1.shape)
print('relation shape:', df_relation.shape)

display(df_1.head())
display(df_relation.head())

overdue shape: (242287, 20)
relation shape: (886, 2)


,order_sn,platform_order_sn,id_card_no_md5,order_time,action_type,action_date,start_date,end_date,original_instalment_qty,rent_days,bill_start_date,bill_end_date,bill_installment_num,expected_pay_date,actual_pay_date,min_1_inst,min_7_inst,min_14_inst,min_30_inst,overdue_days
0,2024110611184925,2024110611184925,83a69bd7cce9258b941a502250b64c97,2024-11-06 11:18:00,NaN,NaT,2024-11-10,2026-05-15,12,552,2024-11-10,2024-12-09,1,2024-11-10,2024-11-06,3.0,6.0,6.0,6.0,-4
1,2024110611184925,2024110611184925,83a69bd7cce9258b941a502250b64c97,2024-11-06 11:18:00,NaN,NaT,2024-11-10,2026-05-15,12,552,2024-12-10,2025-01-09,2,2024-12-09,2024-12-09,3.0,6.0,6.0,6.0,0
2,2024110611184925,2024110611184925,83a69bd7cce9258b941a502250b64c97,2024-11-06 11:18:00,NaN,NaT,2024-11-10,2026-05-15,12,552,2025-01-10,2025-02-09,3,2025-01-09,2025-01-10,3.0,6.0,6.0,6.0,1
3,2024110611184925,2024110611184925,83a69bd7cce9258b941a502250b64c97,2024-11-06 11:18:00,NaN,NaT,2024-11-10,2026-05-15,12,552,2025-02-10,2025-03-09,4,2025-02-09,2025-02-09,3.0,6.0,6.0,6.0,0
4,2024110611184925,2024110611184925,83a69bd7cce9258b941a502250b64c97,2024-11-06 11:18:00,NaN,NaT,2024-11-10,2026-05-15,12,552,2025-03-10,2025-04-09,5,2025-03-09,2025-03-10,3.0,6.0,6.0,6.0,1


,parent_order_sn,sub_order_sn
0,2026012510041707,2026050923162162
1,2025050512003507,2026050811112653
2,2025062015381445,2026042619431838
3,2025050111453106,2026042803103372
4,2025040616261394,2026030316244422


In [3]:
required_overdue_cols = {
    'order_sn',
    'id_card_no_md5',
    'order_time',
    'action_type',
    'action_date',
    'start_date',
    'end_date',
    'rent_days',
    'bill_installment_num',
    'expected_pay_date',
    'actual_pay_date',
    'min_30_inst',
}
required_relation_cols = {'parent_order_sn', 'sub_order_sn'}

missing_overdue_cols = sorted(required_overdue_cols - set(df_1.columns))
missing_relation_cols = sorted(required_relation_cols - set(df_relation.columns))

if missing_overdue_cols:
    raise ValueError(f'overdue file missing columns: {missing_overdue_cols}')
if missing_relation_cols:
    raise ValueError(f'relation file missing columns: {missing_relation_cols}')

In [4]:
# 1. Build root_order_sn for each order chain.
df_1 = df_1.copy()
df_relation = df_relation.copy()

df_1['order_sn'] = df_1['order_sn'].astype('string').str.strip()
df_relation['parent_order_sn'] = df_relation['parent_order_sn'].astype('string').str.strip()
df_relation['sub_order_sn'] = df_relation['sub_order_sn'].astype('string').str.strip()

rel = (
    df_relation[['parent_order_sn', 'sub_order_sn']]
    .dropna()
    .drop_duplicates()
)

sub_parent_cnt = rel.groupby('sub_order_sn')['parent_order_sn'].nunique()
conflict_sub_order_sn = sub_parent_cnt[sub_parent_cnt.gt(1)].index

if len(conflict_sub_order_sn) > 0:
    conflict_rows = (
        rel.loc[rel['sub_order_sn'].isin(conflict_sub_order_sn)]
        .sort_values(['sub_order_sn', 'parent_order_sn'])
    )
    raise ValueError(
        'One sub_order_sn maps to multiple parent_order_sn. Sample conflicts: '
        f"{conflict_rows.head(20).to_dict('records')}"
    )

parent_map = dict(zip(rel['sub_order_sn'], rel['parent_order_sn']))


def find_root_order(order_sn):
    cur = str(order_sn)
    seen = set()

    while cur in parent_map:
        if cur in seen:
            raise ValueError(f'order chain cycle found: {order_sn}')
        seen.add(cur)
        cur = parent_map[cur]

    return cur


all_order_sn = pd.Series(
    pd.concat([
        df_1['order_sn'],
        rel['parent_order_sn'],
        rel['sub_order_sn'],
    ]).dropna().unique(),
    name='order_sn',
)

order_chain_map = pd.DataFrame({'order_sn': all_order_sn})
order_chain_map['root_order_sn'] = order_chain_map['order_sn'].apply(find_root_order)

df_1_chain = df_1.merge(order_chain_map, on='order_sn', how='left')
df_1_chain['root_order_sn'] = df_1_chain['root_order_sn'].fillna(df_1_chain['order_sn'])

print('root order count:', df_1_chain['root_order_sn'].nunique())
display(df_1_chain[['order_sn', 'root_order_sn']].drop_duplicates().head())

root order count: 16683


,order_sn,root_order_sn
0,2024110611184925,2024110611184925
18,2024110612264160,2024110612264160
30,2024110612284259,2024110612284259
42,2024110612314383,2024110612314383
54,2024110612404850,2024110612404850


In [11]:
df_1_chain.to_excel(r'../data/root_order_overdue.xlsx',index=False)
print('保存成功')

In [12]:
# 2. Sort bills by root order chain and recode bill number.
df = df_1_chain.copy()

df['root_order_sn'] = df['root_order_sn'].astype('string').str.strip()
df['order_sn'] = df['order_sn'].astype('string').str.strip()

for col in ['start_date', 'end_date', 'expected_pay_date', 'actual_pay_date', 'order_time', 'action_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

df['bill_installment_num_raw'] = pd.to_numeric(df['bill_installment_num'], errors='coerce')
df['min_30_inst'] = pd.to_numeric(df['min_30_inst'], errors='coerce')

sort_cols = [
    'root_order_sn',
    'start_date',
    'expected_pay_date',
    'end_date',
    'order_time',
    'order_sn',
]

df_chain_recode = (
    df.sort_values(
        by=sort_cols,
        ascending=[True] * len(sort_cols),
        na_position='last',
    )
    .copy()
)

df_chain_recode['chain_bill_installment_num'] = (
    df_chain_recode
    .groupby('root_order_sn')
    .cumcount()
    .add(1)
)

# Locate the original bill row hit by min_30_inst, then map it onto chain order.
df_chain_recode['is_min_30_inst_bill'] = (
    df_chain_recode['min_30_inst'].notna()
    & df_chain_recode['bill_installment_num_raw'].eq(df_chain_recode['min_30_inst'])
).astype(int)

display(
    df_chain_recode.loc[df_chain_recode['is_min_30_inst_bill'].eq(1), [
        'root_order_sn', 'order_sn', 'bill_installment_num_raw',
        'chain_bill_installment_num', 'expected_pay_date', 'min_30_inst'
    ]].head()
)

,root_order_sn,order_sn,bill_installment_num_raw,chain_bill_installment_num,expected_pay_date,min_30_inst
5,2024110611184925,2024110611184925,6,6,2025-04-09,6.0
149,2024111113534788,2024111113534788,5,5,2025-03-13,5.0
175,2024111123103945,2024111123103945,13,13,2025-11-29,13.0
250,2024111214354887,2024111214354887,13,13,2025-11-15,13.0
269,2024111217222172,2024111217222172,11,11,2025-09-15,11.0


In [13]:
# 3. Calculate total rent days by root order chain and keep root_rent_days >= 176.
df = df_chain_recode.copy()

order_rent_days = (
    df[['root_order_sn', 'order_sn', 'rent_days']]
    .drop_duplicates(subset=['root_order_sn', 'order_sn'])
)

root_rent_days = (
    order_rent_days
    .groupby('root_order_sn', as_index=False)['rent_days']
    .sum()
    .rename(columns={'rent_days': 'root_rent_days'})
)

root_rent_days['is_rent_days_ge_176'] = root_rent_days['root_rent_days'].ge(176).astype(int)

df = df.merge(root_rent_days, on='root_order_sn', how='left')
df = df.loc[df['is_rent_days_ge_176'].eq(1)].copy()

print('filtered root order count:', df['root_order_sn'].nunique())
print('filtered bill rows:', len(df))

filtered root order count: 16271
filtered bill rows: 241160


In [14]:
# 4. Create MOB180 DPD30 label from min_30_inst.
df = df.copy()

df['root_start_date'] = df.groupby('root_order_sn')['start_date'].transform('min')
df['mob_end'] = df['root_start_date'] + pd.Timedelta(days=MOB_DAYS)
df['label_mature_date'] = df['mob_end'] + pd.Timedelta(days=DPD_DAYS)

df['in_mob_bill'] = (
    df['expected_pay_date'].notna()
    & df['expected_pay_date'].le(df['mob_end'])
).astype(int)

# target=1 if the min_30_inst hit bill is inside the MOB window.
df['hit_min_30_inst'] = (
    df['in_mob_bill'].eq(1)
    & df['is_min_30_inst_bill'].eq(1)
).astype(int)

df['target_raw'] = df.groupby('root_order_sn')['hit_min_30_inst'].transform('max')

display(
    df.loc[df['hit_min_30_inst'].eq(1), [
        'root_order_sn', 'order_sn', 'bill_installment_num_raw',
        'chain_bill_installment_num', 'expected_pay_date', 'min_30_inst',
        'root_start_date', 'mob_end'
    ]].head()
)

,root_order_sn,order_sn,bill_installment_num_raw,chain_bill_installment_num,expected_pay_date,min_30_inst,root_start_date,mob_end
5,2024110611184925,2024110611184925,6,6,2025-04-09,6.0,2024-11-10,2025-05-09
149,2024111113534788,2024111113534788,5,5,2025-03-13,5.0,2024-11-14,2025-05-13
277,2024111220492432,2024111220492432,6,6,2025-04-14,6.0,2024-11-15,2025-05-14
391,2024111315281681,2024111315281681,4,4,2025-02-16,4.0,2024-11-17,2025-05-16
409,2024111315472925,2024111315472925,3,3,2025-01-16,3.0,2024-11-17,2025-05-16


In [15]:
# 5. Aggregate to root-order level and export all labels first.
early_close_actions = ['early_return', 'early_buyout']
df['is_early_closed_row'] = df['action_type'].isin(early_close_actions).astype(int)

root_status = (
    df.groupby('root_order_sn')
      .agg(
          id_card_no_md5=('id_card_no_md5', 'first'),
          root_order_time=('order_time', 'min'),
          target_raw=('hit_min_30_inst', 'max'),
          root_start_date=('root_start_date', 'min'),
          mob_end=('mob_end', 'min'),
          label_mature_date=('label_mature_date', 'min'),
          root_rent_days=('root_rent_days', 'max'),
          min_30_inst_min=('min_30_inst', 'min'),
          has_early_close=('is_early_closed_row', 'max'),
      )
      .reset_index()
)

root_status['is_mature'] = root_status['label_mature_date'].le(DATA_CUTOFF)
root_status['target'] = root_status['target_raw'].astype('Int64')

model_sample = root_status.copy()

print('all root orders:', len(root_status))
print('export samples:', len(model_sample))
print('mature samples:', int(model_sample['is_mature'].sum()))
print('not mature samples:', int((~model_sample['is_mature']).sum()))
display(root_status['target'].value_counts(dropna=False).rename('cnt').to_frame())

all root orders: 16271
export samples: 16271
mature samples: 7965
not mature samples: 8306


,cnt
target,
0,15524
1,747


In [16]:
# 6. Check target result.
assert model_sample['target'].isna().sum() == 0

summary = pd.DataFrame({
    'sample_cnt': [len(model_sample)],
    'mature_cnt': [int(model_sample['is_mature'].sum())],
    'not_mature_cnt': [int((~model_sample['is_mature']).sum())],
    'bad_cnt': [int(model_sample['target'].sum())],
    'bad_rate': [model_sample['target'].mean()],
    'data_cutoff': [DATA_CUTOFF],
    'mob_days': [MOB_DAYS],
    'dpd_days': [DPD_DAYS],
})

display(summary)
display(model_sample.head())

,sample_cnt,mature_cnt,not_mature_cnt,bad_cnt,bad_rate,data_cutoff,mob_days,dpd_days
0,16271,7965,8306,747,0.04591,2026-05-11,180,30


,root_order_sn,id_card_no_md5,root_order_time,target_raw,root_start_date,mob_end,label_mature_date,root_rent_days,min_30_inst_min,has_early_close,is_mature,target
0,2024110611184925,83a69bd7cce9258b941a502250b64c97,2024-11-06 11:18:00,1,2024-11-10,2025-05-09,2025-06-08,552,6.0,0,True,1
1,2024110612264160,2b39a24601453b25bcfd7dec04d810b8,2024-11-06 12:26:00,0,2024-11-10,2025-05-09,2025-06-08,365,NaN,1,True,0
2,2024110612284259,18c3dfc9c2289cfcd8060f797017b043,2024-11-06 12:28:00,0,2024-11-10,2025-05-09,2025-06-08,365,NaN,1,True,0
3,2024110612314383,7304bf30408883e590863f0a0bcf2848,2024-11-06 12:31:00,0,2024-11-15,2025-05-14,2025-06-13,365,NaN,1,True,0
4,2024110612404850,333a3f48fd0c311db87bff4ae577374c,2024-11-06 12:40:00,0,2024-11-10,2025-05-09,2025-06-08,365,NaN,1,True,0


In [17]:
# 7. Save result.
output_cols = [
    'root_order_sn',
    'id_card_no_md5',
    'root_order_time',
    'root_start_date',
    'root_rent_days',
    'mob_end',
    'label_mature_date',
    'is_mature',
    'min_30_inst_min',
    'target',
    'has_early_close',
]

save_file = OUTPUT_FILE
try:
    model_sample[output_cols].to_excel(save_file, index=False)
except PermissionError:
    save_file = OUTPUT_FILE.with_name(f'{OUTPUT_FILE.stem}_new{OUTPUT_FILE.suffix}')
    model_sample[output_cols].to_excel(save_file, index=False)
    print('target file was locked; saved fallback file:', save_file.name)
else:
    print('saved:', save_file.name)

saved: user_target_min_30_inst.xlsx


### 取未表现m180+的订单做验证

In [18]:
# 8. 按订单展开每一期 DPD15 ever 表现。
# 口径：当期 overdue_days >= 15 记为当期命中；截至当前期任意一期命中过，则当前期及后续期 ever15 = 1。
# 未到 15 天观察窗口的期数，如果前面没有命中过，则 ever15 留空；如果前面已命中，则 ever15 仍为 1。
EVER_DPD_DAYS = 15
OUTPUT_INST_EVER15_FILE = DATA_DIR / 'user_target_inst_ever15.xlsx'

inst_long = df.copy()

for col in ['expected_pay_date', 'actual_pay_date', 'order_time', 'start_date', 'end_date']:
    if col in inst_long.columns:
        inst_long[col] = pd.to_datetime(inst_long[col], errors='coerce')

inst_long['chain_bill_installment_num'] = pd.to_numeric(inst_long['chain_bill_installment_num'], errors='coerce').astype('Int64')
inst_long['overdue_days'] = pd.to_numeric(inst_long['overdue_days'], errors='coerce')
inst_long['inst_mature_date'] = inst_long['expected_pay_date'] + pd.Timedelta(days=EVER_DPD_DAYS)
inst_long['is_inst_mature'] = inst_long['inst_mature_date'].le(DATA_CUTOFF)

# 当期 DPD15 标签：只有该期已经过了 15 天观察窗口，才给 0/1；未观察充分则留空。
inst_long['dpd15_current'] = pd.Series(pd.NA, index=inst_long.index, dtype='Int64')
observed_mask = inst_long['is_inst_mature'].eq(True)
inst_long.loc[observed_mask, 'dpd15_current'] = inst_long.loc[observed_mask, 'overdue_days'].ge(EVER_DPD_DAYS).astype('int64')

# ever DPD15：按订单、期数累计。前面任意一期命中后，后续期数全部为 1。
inst_long = inst_long.sort_values([
    'root_order_sn',
    'chain_bill_installment_num',
    'expected_pay_date',
    'order_sn',
]).copy()

inst_long['dpd15_hit_for_cum'] = inst_long['dpd15_current'].fillna(0).astype('int64')
inst_long['ever15_cum'] = inst_long.groupby('root_order_sn')['dpd15_hit_for_cum'].cummax()
inst_long['ever15_target'] = pd.Series(pd.NA, index=inst_long.index, dtype='Int64')
inst_long.loc[inst_long['ever15_cum'].eq(1), 'ever15_target'] = 1
inst_long.loc[inst_long['ever15_cum'].eq(0) & observed_mask, 'ever15_target'] = 0

inst_output_cols = [
    'root_order_sn', 'order_sn', 'id_card_no_md5', 'root_order_time',
    'root_start_date', 'root_rent_days', 'chain_bill_installment_num',
    'bill_installment_num_raw', 'expected_pay_date', 'actual_pay_date',
    'inst_mature_date', 'is_inst_mature', 'overdue_days',
    'dpd15_current', 'ever15_target', 'has_early_close',
]
inst_output_cols = [c for c in inst_output_cols if c in inst_long.columns]
inst_long_out = inst_long[inst_output_cols].copy()

# 按订单展开为宽表：每个 root_order_sn 一行，每一期一个 ever15 字段。
inst_wide_ever15 = (
    inst_long_out
    .pivot_table(
        index='root_order_sn',
        columns='chain_bill_installment_num',
        values='ever15_target',
        aggfunc='max',
        dropna=False,
    )
    .rename(columns=lambda x: f'ever15_inst_{int(x)}')
    .reset_index()
)

inst_wide_current = (
    inst_long_out
    .pivot_table(
        index='root_order_sn',
        columns='chain_bill_installment_num',
        values='dpd15_current',
        aggfunc='max',
        dropna=False,
    )
    .rename(columns=lambda x: f'dpd15_inst_{int(x)}')
    .reset_index()
)

order_base_cols = [
    'root_order_sn', 'id_card_no_md5', 'root_order_time', 'root_start_date',
    'root_rent_days', 'mob_end', 'label_mature_date', 'is_mature',
    'target', 'has_early_close',
]
order_base_cols = [c for c in order_base_cols if c in root_status.columns]
inst_wide_out = (
    root_status[order_base_cols]
    .merge(inst_wide_ever15, on='root_order_sn', how='left')
    .merge(inst_wide_current, on='root_order_sn', how='left')
)

# 只验证 MOB180 DPD30 尚未表现到的订单；已经表现到的订单已用于建模，不放进验证结果。
verify_root_orders = root_status.loc[root_status['is_mature'].eq(False), 'root_order_sn']
verify_inst_long = inst_long_out.loc[inst_long_out['root_order_sn'].isin(verify_root_orders)].copy()
verify_inst_wide = inst_wide_out.loc[inst_wide_out['root_order_sn'].isin(verify_root_orders)].copy()
verify_ever15_order_cnt = int(
    inst_long.loc[inst_long['root_order_sn'].isin(verify_root_orders)]
    .groupby('root_order_sn')['ever15_cum']
    .max()
    .sum()
)

inst_summary = pd.DataFrame({
    'verify_root_order_cnt': [verify_root_orders.nunique()],
    'verify_bill_row_cnt': [len(verify_inst_long)],
    'verify_observed_bill_cnt': [int(verify_inst_long['is_inst_mature'].sum())],
    'verify_dpd15_current_hit_cnt': [int(verify_inst_long['dpd15_current'].fillna(0).sum())],
    'verify_ever15_root_order_cnt': [verify_ever15_order_cnt],
    'data_cutoff': [DATA_CUTOFF],
    'ever_dpd_days': [EVER_DPD_DAYS],
})

display(inst_summary)
display(verify_inst_long.head(20))
display(verify_inst_wide.head())

# save_file = OUTPUT_INST_EVER15_FILE
# try:
#     with pd.ExcelWriter(save_file) as writer:
#         inst_summary.to_excel(writer, sheet_name='summary', index=False)
#         verify_inst_long.to_excel(writer, sheet_name='inst_long_verify', index=False)
#         verify_inst_wide.to_excel(writer, sheet_name='order_wide_verify', index=False)
# except PermissionError:
#     save_file = OUTPUT_INST_EVER15_FILE.with_name(f'{OUTPUT_INST_EVER15_FILE.stem}_new{OUTPUT_INST_EVER15_FILE.suffix}')
#     with pd.ExcelWriter(save_file) as writer:
#         inst_summary.to_excel(writer, sheet_name='summary', index=False)
#         verify_inst_long.to_excel(writer, sheet_name='inst_long_verify', index=False)
#         verify_inst_wide.to_excel(writer, sheet_name='order_wide_verify', index=False)
#     print('inst ever15 file was locked; saved fallback file:', save_file.name)
# else:
#     print('saved:', save_file.name)


,verify_root_order_cnt,verify_bill_row_cnt,verify_observed_bill_cnt,verify_dpd15_current_hit_cnt,verify_ever15_root_order_cnt,data_cutoff,ever_dpd_days
0,8306,116816,27343,369,192,2026-05-11,15


,root_order_sn,order_sn,id_card_no_md5,root_start_date,root_rent_days,chain_bill_installment_num,bill_installment_num_raw,expected_pay_date,actual_pay_date,inst_mature_date,is_inst_mature,overdue_days,dpd15_current,ever15_target
122438,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,1,1,2025-10-15,2025-09-26,2025-10-30,True,-19,0,0
122439,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,2,2,2025-11-14,2025-10-17,2025-11-29,True,-28,0,0
122440,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,3,3,2025-12-14,2025-11-18,2025-12-29,True,-26,0,0
122441,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,4,4,2026-01-14,2025-12-18,2026-01-29,True,-27,0,0
122442,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,5,5,2026-02-14,2025-12-18,2026-03-01,True,-58,0,0
122443,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,6,6,2026-03-14,2025-12-18,2026-03-29,True,-86,0,0
122444,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,7,7,2026-04-14,2026-03-18,2026-04-29,True,-27,0,0
122445,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,8,8,2026-05-14,2026-04-17,2026-05-29,False,-27,<NA>,<NA>
122446,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,9,9,2026-06-14,NaT,2026-06-29,False,0,<NA>,<NA>
122447,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-10-15,365,10,10,2026-07-14,NaT,2026-07-29,False,0,<NA>,<NA>


,root_order_sn,id_card_no_md5,root_order_time,root_start_date,root_rent_days,mob_end,label_mature_date,is_mature,target,has_early_close,...,dpd15_inst_410,dpd15_inst_411,dpd15_inst_412,dpd15_inst_413,dpd15_inst_414,dpd15_inst_415,dpd15_inst_416,dpd15_inst_417,dpd15_inst_418,dpd15_inst_419
7782,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,2025-10-15,365,2026-04-13,2026-05-13,False,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7794,2025092801102263,2d57735e3902de11c6a3166a1f867684,2025-09-28 01:10:00,2025-10-15,365,2026-04-13,2026-05-13,False,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7795,2025092801503598,a21a22031ec62edcd8308ed62f510c86,2025-09-28 01:50:00,2025-10-15,365,2026-04-13,2026-05-13,False,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7796,2025092806384436,62c6b6d9c111c2c3a829213e97e3ea51,2025-09-28 06:38:00,2025-10-15,365,2026-04-13,2026-05-13,False,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7801,2025092810184433,29b64a97f833f660cd466c85d3296d86,2025-09-28 10:18:00,2025-10-15,365,2026-04-13,2026-05-13,False,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


saved: user_target_inst_ever15.xlsx


####

In [8]:
df = pd.read_excel(r'../data/root_order_overdue.xlsx')
# =========================
# 参数区
# =========================
data_cutoff = pd.Timestamp("2026-05-11")   # 数据观察截止日，按你的实际取数截止日改
mob_days = 180
dpd30_days = 30
dpd15_days = 15

# True：只保留 expected_pay_date + 15 已经到期的期数
# False：保留所有期数，但会输出 inst_dpd15_mature 标识
only_mature_inst = False


# =========================
# 1. 复制数据 + 字段标准化
# =========================
df_check = df.copy()
df_check.columns = df_check.columns.astype(str).str.strip()

date_cols = [
    "order_time","order_time", "action_date", "start_date", "end_date",
    "bill_start_date", "bill_end_date",
    "expected_pay_date", "actual_pay_date"
]

for c in date_cols:
    if c in df_check.columns:
        df_check[c] = pd.to_datetime(df_check[c], errors="coerce")

num_cols = ["bill_installment_num", "overdue_days"]
for c in num_cols:
    if c in df_check.columns:
        df_check[c] = pd.to_numeric(df_check[c], errors="coerce")

required_cols = [
    "root_order_sn",
    "order_sn",
    "order_time",
    "start_date",
    "bill_start_date",
    "bill_end_date",
    "bill_installment_num",
    "expected_pay_date",
    "overdue_days"
]

missing_cols = [c for c in required_cols if c not in df_check.columns]
if missing_cols:
    raise ValueError(f"缺少必要字段: {missing_cols}")


# =========================
# 2. 计算 MOB180 DPD30 是否已经表现到
#    表现到期日 = root_start_date + 180 + 30
# =========================
df_check["root_start_date"] = df_check.groupby("root_order_sn")["start_date"].transform("min")

df_check["mob180_end"] = df_check["root_start_date"] + pd.Timedelta(days=mob_days)
df_check["mob180_dpd30_mature_date"] = df_check["mob180_end"] + pd.Timedelta(days=dpd30_days)

# 只拿 MOB180 DPD30 没有表现到的订单
# 也就是：mob180_dpd30_mature_date > data_cutoff
df_check["mob180_dpd30_not_mature"] = (
    df_check["mob180_dpd30_mature_date"] > data_cutoff
).astype(int)

valid_root = (
    df_check.loc[df_check["mob180_dpd30_not_mature"].eq(1), "root_order_sn"]
    .dropna()
    .unique()
)

df_valid = df_check[df_check["root_order_sn"].isin(valid_root)].copy()


# =========================
# 3. 按 root_order_sn 重排链路期数
#    不再直接相信原来的 bill_installment_num
# =========================
sort_cols = [
    "root_order_sn",
    "expected_pay_date",
    "bill_start_date",
    "bill_end_date",
    "order_sn",
    "order_time",
    "bill_installment_num"
]

df_valid = df_valid.sort_values(sort_cols).reset_index(drop=True)

df_valid["chain_bill_installment_num"] = (
    df_valid.groupby("root_order_sn").cumcount() + 1
)
# =========================
# 3.1 只保留链路前12期
#     第13期及以后全部不看
# =========================
max_chain_inst = 12

df_valid = df_valid[
    df_valid["chain_bill_installment_num"].le(max_chain_inst)
].copy()
# =========================
# 4. 每一期输出 expected_pay_date + 15天
#    验证场景：用数据截止日 data_cutoff 作为观察时点
# =========================
df_valid["expected_pay_date_plus15"] = (
    df_valid["expected_pay_date"] + pd.Timedelta(days=dpd15_days)
)

# 验证表现用数据截止日，不用订单 order_time
df_valid["obs_cutoff_date"] = data_cutoff

# 该期 DPD15 是否已经表现完
df_valid["inst_dpd15_mature"] = (
    df_valid["expected_pay_date_plus15"] <= df_valid["obs_cutoff_date"]
).astype("Int64")


# =========================
# 5. 当前期是否 DPD15+
#    没表现完的期数必须是 NA，不能是 0
# =========================
df_valid["current_dpd15_flag"] = pd.Series(pd.NA, index=df_valid.index, dtype="Int64")

mature_mask = df_valid["inst_dpd15_mature"].eq(1)

df_valid.loc[mature_mask, "current_dpd15_flag"] = (
    df_valid.loc[mature_mask, "overdue_days"] >= dpd15_days
).astype("Int64")


# =========================
# 6. 截至当前期是否 ever DPD15+
#    前面一期命中后，后续期数 ever 全部为 1
# =========================
def calc_ever_dpd15(g):
    g = g.sort_values("chain_bill_installment_num").copy()

    # 当前期命中累计
    hit_cum = g["current_dpd15_flag"].fillna(0).cummax()

    out = hit_cum.astype("Int64")

    # 没表现完，并且前面没命中过，不能给 0，给 NA
    not_mature_no_hit = g["inst_dpd15_mature"].ne(1) & hit_cum.eq(0)
    out.loc[not_mature_no_hit] = pd.NA

    return out


df_valid["ever_dpd15_flag"] = (
    df_valid
    .groupby("root_order_sn", group_keys=False)
    .apply(calc_ever_dpd15)
)

# =========================
# 7. 输出明细结果：一行 = 一个订单的一期
# =========================
keep_cols = [
    "root_order_sn",
    "order_sn",
    "id_card_no_md5",
    "order_time",
    "chain_bill_installment_num",
    "bill_installment_num",
    "expected_pay_date",
    "expected_pay_date_plus15",
    "actual_pay_date",
    "overdue_days",
    "inst_dpd15_mature",
    "current_dpd15_flag",
    "ever_dpd15_flag",
    "root_start_date",
    "mob180_end",
    "mob180_dpd30_mature_date",
    "mob180_dpd30_not_mature",
    "action_type",
    "action_date"
]

keep_cols = [c for c in keep_cols if c in df_valid.columns]

df_result_long = df_valid[keep_cols].copy()

if only_mature_inst:
    df_result_long = df_result_long[df_result_long["inst_dpd15_mature"].eq(1)].copy()

df_result_long = df_result_long.sort_values(
    ["root_order_sn", "chain_bill_installment_num"]
).reset_index(drop=True)

print("明细结果行数:", len(df_result_long))
print("订单数:", df_result_long["root_order_sn"].nunique())

df_result_long.head(50)

明细结果行数: 99222
订单数: 8539


C:\Users\Administrator\AppData\Local\Temp\ipykernel_28108\3331144395.py:159: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_valid


,root_order_sn,order_sn,id_card_no_md5,order_time,chain_bill_installment_num,bill_installment_num,expected_pay_date,expected_pay_date_plus15,actual_pay_date,overdue_days,inst_dpd15_mature,current_dpd15_flag,ever_dpd15_flag,root_start_date,mob180_end,mob180_dpd30_mature_date,mob180_dpd30_not_mature,action_type,action_date
0,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,1,1,2025-10-15,2025-10-30,2025-09-26,-19,1,0,0,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
1,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,2,2,2025-11-14,2025-11-29,2025-10-17,-28,1,0,0,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
2,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,3,3,2025-12-14,2025-12-29,2025-11-18,-26,1,0,0,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
3,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,4,4,2026-01-14,2026-01-29,2025-12-18,-27,1,0,0,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
4,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,5,5,2026-02-14,2026-03-01,2025-12-18,-58,1,0,0,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
5,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,6,6,2026-03-14,2026-03-29,2025-12-18,-86,1,0,0,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
6,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,7,7,2026-04-14,2026-04-29,2026-03-18,-27,1,0,0,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
7,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,8,8,2026-05-14,2026-05-29,2026-04-17,-27,0,<NA>,<NA>,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
8,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,9,9,2026-06-14,2026-06-29,NaT,0,0,<NA>,<NA>,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT
9,2025092623391573,2025092623391573,ab9711da199a2d4918769ef1890181a1,2025-09-26 23:39:00,10,10,2026-07-14,2026-07-29,NaT,0,0,<NA>,<NA>,2025-10-15,2026-04-13,2026-05-13,1,NaN,NaT


In [30]:
# =========================
# 8. 转宽表：每个 root_order_sn 一行
# =========================
wide_mature = df_result_long.pivot_table(
    index="root_order_sn",
    columns="chain_bill_installment_num",
    values="inst_dpd15_mature",
    aggfunc="first"
)

wide_mature.columns = [
    f"inst_{int(c)}_dpd15_mature" for c in wide_mature.columns
]
# 8.1 每期逾期天数
wide_overdue = df_result_long.pivot_table(
    index="root_order_sn",
    columns="chain_bill_installment_num",
    values="overdue_days",
    aggfunc="first"
)

wide_overdue.columns = [
    f"inst_{int(c)}_overdue_days" for c in wide_overdue.columns
]

# 8.2 每期 expected_pay_date + 15
wide_mature_date = df_result_long.pivot_table(
    index="root_order_sn",
    columns="chain_bill_installment_num",
    values="expected_pay_date_plus15",
    aggfunc="first"
)

wide_mature_date.columns = [
    f"inst_{int(c)}_expected_pay_date_plus15" for c in wide_mature_date.columns
]

# 8.3 每期当期 DPD15
wide_current_dpd15 = df_result_long.pivot_table(
    index="root_order_sn",
    columns="chain_bill_installment_num",
    values="current_dpd15_flag",
    aggfunc="first"
)

wide_current_dpd15.columns = [
    f"inst_{int(c)}_current_dpd15_flag" for c in wide_current_dpd15.columns
]

# 8.4 截至当前期 ever DPD15
wide_ever_dpd15 = df_result_long.pivot_table(
    index="root_order_sn",
    columns="chain_bill_installment_num",
    values="ever_dpd15_flag",
    aggfunc="first"
)

wide_ever_dpd15.columns = [
    f"inst_{int(c)}_ever_dpd15_flag" for c in wide_ever_dpd15.columns
]

# 8.5 订单基础信息
base_info = (
    df_result_long
    .sort_values(["root_order_sn", "chain_bill_installment_num"])
    .groupby("root_order_sn")
    .agg(
        order_sn=("order_sn", "first"),
        order_time=("order_time", "first"),
        id_card_no_md5=("id_card_no_md5", "first"),
        root_start_date=("root_start_date", "first"),
        mob180_end=("mob180_end", "first"),
        mob180_dpd30_mature_date=("mob180_dpd30_mature_date", "first")
    )
)

df_result_wide = (
    base_info
    .join(wide_mature_date)
    .join(wide_mature)
    .join(wide_overdue)
    .join(wide_current_dpd15)
    .join(wide_ever_dpd15)
    .reset_index()
)

print("宽表订单数:", len(df_result_wide))

宽表订单数: 8539


In [32]:
df_result_wide[['root_order_sn','order_time',
                'root_start_date','id_card_no_md5',
                'inst_1_ever_dpd15_flag',
                'inst_2_ever_dpd15_flag',
                'inst_3_ever_dpd15_flag',
                'inst_4_ever_dpd15_flag',
                'inst_5_ever_dpd15_flag',
                'inst_6_ever_dpd15_flag',
                'inst_7_ever_dpd15_flag'
                ]].to_excel(r'../data/oot用户标签验证.xlsx',index=False)
print('保存成功')